# AI fact-checking agent — Version 0

Our workstream only: take **one English sentence**, decide whether it is checkable,
search the web, then compare retrieved evidence with the claim using a local Hugging Face NLI model.
Our teammates handle transcription and storage in their own notebooks.

This is the simple baseline for later comparisons: **no claim/evidence cache or speed-ups**,
no dataclasses, mock models, or test framework. Results are ordinary dictionaries.
Models load once; every checkable claim triggers a fresh search and sequential downloads.

## 1. Select the environment
Set up the Python 3.12 environment from the repo root (see the README):

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

In VS Code, select the `.venv` Python interpreter as this notebook's kernel.
The first model load downloads weights; inference then runs locally. Web search still needs internet.


In [18]:
import json
import math
import os
import re
import statistics
import time

os.environ.setdefault("HF_HOME", os.path.abspath(".cache/huggingface"))

import torch
from ddgs import DDGS
from trafilatura import extract, fetch_url
from transformers import AutoModelForSequenceClassification, AutoTokenizer


## 2. Load the local model
[NLI MiniLM model card](https://huggingface.co/cross-encoder/nli-MiniLM2-L6-H768).
NLI compares an evidence passage (premise) with a claim (hypothesis), returning
`entailment`, `contradiction`, and `neutral` scores. It does not generate an answer.
We also reuse it inside a small hybrid checkability detector; that part needs separate evaluation.

In [19]:
MODEL_NAME = "cross-encoder/nli-MiniLM2-L6-H768"
MAX_RESULTS = 3
MAX_PASSAGES = 5
EVIDENCE_STRENGTH_THRESHOLD = 0.60
CONFLICT_MARGIN = 0.10

startup = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()


def nli(premise, hypothesis):
    inputs = tokenizer(
        premise, hypothesis, return_tensors="pt",
        truncation="only_first", max_length=512,
    )
    with torch.inference_mode():
        scores = model(**inputs).logits.softmax(dim=-1)[0].tolist()
    return {model.config.id2label[i].lower(): score for i, score in enumerate(scores)}


nli("Paris is in France.", "Paris is located in France.")
startup_ms = round((time.perf_counter() - startup) * 1000, 1)
print(f"Model loading + first inference: {startup_ms} ms")

Model loading + first inference: 635.6 ms


## 3. Is the sentence checkable?
The baseline filters obvious questions, commands, and subjective opinions using transparent rules.
The remaining declarative sentences are treated as checkable factual claims. Whether useful
public evidence exists is decided later by retrieval and can produce `uncertain`.
Pass standalone sentences with explicit subjects; pronoun resolution and compound-claim splitting come later.

In [20]:
QUESTION_STARTS = {
    "who", "what", "when", "where", "why", "how", "which",
    "is", "are", "do", "does", "did", "can", "could", "would", "should"
}
OPINION_MARKERS = {
    "i think", "i believe", "i feel", "in my opinion", "i prefer",
    "best", "worst", "beautiful", "ugly", "delicious", "boring", "amazing", "favorite"
}
COMMAND_STARTS = {
    "please", "tell", "give", "show", "explain", "close", "open",
    "stop", "go", "make", "write", "run", "click", "select"
}


def is_checkable(text):
    clean = text.strip()
    words = re.findall(r"[a-z']+", clean.lower())
    first_word = words[0] if words else ""

    if len(words) < 3:
        return False
    if clean.endswith("?") or first_word in QUESTION_STARTS:
        return False
    if first_word in COMMAND_STARTS:
        return False
    if any(marker in clean.lower() for marker in OPINION_MARKERS):
        return False
    return True

## 4. Search and read web pages
`ddgs` provides web metasearch without an API key. This baseline searches the open web;
a source allowlist can be added once the team chooses its topic.
Download up to three pages sequentially, split them into short passages, and select up to
five using word overlap. This only prepares inputs that fit the model.
Downloaded page sentences are preferred. Search-result snippets are retained as labelled
fallback evidence when a page cannot be read or its relevant sentence is not extracted.

In [21]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for",
    "from", "has", "have", "in", "is", "it", "of", "on",
    "that", "the", "to", "was", "were", "with"
}


def content_words(text):
    return set(re.findall(r"\w+", text.lower())) - STOPWORDS


def search_web(claim):
    return list(DDGS(timeout=10).text(claim, max_results=MAX_RESULTS))


def collect_evidence(claim, search_results):
    claim_words = content_words(claim)
    passages = []
    errors = []
    seen = set()

    def add_passage(text, url, source_type):
        passage = re.sub(r"\s+", " ", text).strip()[:700]
        key = passage.lower()
        if not passage or key in seen:
            return
        overlap = len(claim_words & content_words(passage))
        relevance = overlap / max(len(claim_words), 1)
        if relevance >= 0.5:
            seen.add(key)
            passages.append({
                "text": passage, "url": url, "source_type": source_type,
                "overlap": overlap, "relevance": round(relevance, 3),
            })

    for result in search_results:
        url = result.get("href") or result.get("url")
        if not url:
            continue
        snippet = result.get("body") or result.get("snippet") or ""
        if snippet:
            add_passage(snippet, url, "search_snippet")
        try:
            html = fetch_url(url)
            text = extract(html, include_comments=False) if html else None
        except Exception as error:
            errors.append(f"{url}: {error}")
            continue
        if not text:
            errors.append(f"{url}: no readable text")
            continue
        sentences = re.split(r"(?<=[.!?])\s+", re.sub(r"\s+", " ", text))
        for sentence in sentences:
            add_passage(sentence, url, "page")
    passages.sort(
        key=lambda passage: (passage["relevance"], passage["source_type"] == "page"),
        reverse=True,
    )
    return passages[:MAX_PASSAGES], errors

## 5. Check the claim
Return `not_checkable`, `supported`, `contradicted`, or `uncertain`, plus evidence and stage timings.
If strong support and strong contradiction both appear, return `uncertain`.
Missing evidence also means `uncertain`. An NLI verdict describes the retrieved text;
it does not establish that a source is trustworthy. Review the citations when evaluating quality.

In [22]:
def check_claim(text):
    text = re.sub(r"\b(\w+)(?:\s+\1\b)+", r"\1", text.strip(), flags=re.IGNORECASE)
    started = time.perf_counter()
    result = {"claim": text, "verdict": "not_checkable", "evidence": [],
              "errors": [], "timings_ms": {}}
    timings = result["timings_ms"]
    checkable = is_checkable(text)
    timings["checkability"] = (time.perf_counter() - started) * 1000

    if checkable:
        result["verdict"] = "uncertain"
        stage = time.perf_counter()
        try:
            hits = search_web(text)
        except Exception as error:
            hits = []
            result["errors"].append(f"Search failed: {error}")
        timings["search"] = (time.perf_counter() - stage) * 1000

        stage = time.perf_counter()
        passages, errors = collect_evidence(text, hits)
        result["errors"].extend(errors)
        timings["retrieval"] = (time.perf_counter() - stage) * 1000

        stage = time.perf_counter()
        for passage in passages:
            passage["scores"] = nli(passage["text"], text)
            passage["support_strength"] = round(
                passage["scores"]["entailment"] * passage["relevance"], 4
            )
            passage["contradiction_strength"] = round(
                passage["scores"]["contradiction"] * passage["relevance"], 4
            )
        if passages:
            support = max(passages, key=lambda p: p["support_strength"])
            against = max(passages, key=lambda p: p["contradiction_strength"])
            support_score = support["support_strength"]
            contradiction_score = against["contradiction_strength"]
            supported = support_score >= EVIDENCE_STRENGTH_THRESHOLD
            contradicted = contradiction_score >= EVIDENCE_STRENGTH_THRESHOLD
            close_conflict = abs(support_score - contradiction_score) < CONFLICT_MARGIN
            if supported and contradicted and close_conflict:
                result["evidence"] = [support, against]
            elif supported and support_score > contradiction_score:
                result.update(verdict="supported", evidence=[support])
            elif contradicted:
                result.update(verdict="contradicted", evidence=[against])
            else:
                result["evidence"] = passages
        timings["verification"] = (time.perf_counter() - stage) * 1000

    timings["total"] = (time.perf_counter() - started) * 1000
    result["timings_ms"] = {name: round(ms, 2) for name, ms in timings.items()}
    return result

## 6. Small baseline benchmark
Each example is checked once. `not_checkable` examples evaluate the detector, while the
other examples evaluate retrieval and verification.

**p50** is the median latency. **p95** is the time within which roughly 95% of requests finish.
Model startup is reported separately from request latency.

In [23]:
examples = [
    ("Switzerland has 26 cantons.", "supported"),
    ("Paris is the capital of Germany.", "contradicted"),
    ("The Moon is a natural satellite of Earth.", "supported"),
    ("I think Switzerland is beautiful.", "not_checkable"),
    ("What is the capital of France?", "not_checkable"),
]

benchmark_results = []
for text, expected in examples:
    output = check_claim(text)
    output["expected"] = expected
    benchmark_results.append(output)
    print(output["verdict"], "|", output["timings_ms"]["total"], "ms |", text)


def latency_summary(results):
    values = sorted(row["timings_ms"]["total"] for row in results)
    return {
        "count": len(values),
        "p50_ms": statistics.median(values),
        "p95_ms": values[math.ceil(0.95 * len(values)) - 1],
        "under_1_second_percent": 100 * sum(ms < 1000 for ms in values) / len(values),
    }


checkable_results = [r for r in benchmark_results if r["expected"] != "not_checkable"]
summary = {
    "model_startup_ms": startup_ms,
    "all_inputs": latency_summary(benchmark_results),
    "checkable_claims": latency_summary(checkable_results),
    "checkability_accuracy": statistics.mean(
        (r["verdict"] != "not_checkable") == (r["expected"] != "not_checkable")
        for r in benchmark_results
    ),
    "claim_verdict_accuracy": statistics.mean(
        r["verdict"] == r["expected"] for r in checkable_results
    ),
}
print(json.dumps(summary, indent=2))

supported | 2467.8 ms | Switzerland has 26 cantons.
contradicted | 5394.73 ms | Paris is the capital of Germany.
supported | 4973.44 ms | The Moon is a natural satellite of Earth.
not_checkable | 0.03 ms | I think Switzerland is beautiful.
not_checkable | 0.01 ms | What is the capital of France?
{
  "model_startup_ms": 635.6,
  "all_inputs": {
    "count": 5,
    "p50_ms": 2467.8,
    "p95_ms": 5394.73,
    "under_1_second_percent": 40.0
  },
  "checkable_claims": {
    "count": 3,
    "p50_ms": 4973.44,
    "p95_ms": 5394.73,
    "under_1_second_percent": 0.0
  },
  "checkability_accuracy": 1,
  "claim_verdict_accuracy": 1
}


## 7. Inspect one result
Display the complete evidence and timing output for the first benchmark claim without
performing another web search. Change the list index to inspect a different claim.

In [24]:
print(json.dumps(benchmark_results[0], indent=2))

{
  "claim": "Switzerland has 26 cantons.",
  "verdict": "supported",
  "evidence": [
    {
      "text": "The 26 cantons of Switzerland[note 1] are the member states of the Swiss Confederation. The nucleus of the Swiss Confederacy in the form of the first three confederate allies used to be referred to as the Waldst\u00e4tte.",
      "url": "https://en.wikipedia.org/wiki/Cantons_of_Switzerland",
      "source_type": "search_snippet",
      "overlap": 3,
      "relevance": 1.0,
      "scores": {
        "contradiction": 0.0010682886932045221,
        "entailment": 0.9938662648200989,
        "neutral": 0.005065358709543943
      },
      "support_strength": 0.9939,
      "contradiction_strength": 0.0011
    }
  ],
  "errors": [],
  "timings_ms": {
    "checkability": 0.03,
    "search": 1181.78,
    "retrieval": 1097.28,
    "verification": 188.7,
    "total": 2467.8
  },
  "expected": "supported"
}


Before comparing optimisations, record the hardware, package/model versions, date, retrieval failures,
and manually review the cited evidence. Use a larger fixed dataset including insufficient-evidence cases.
The result dictionaries keep stage timings for further analysis.

Next experiments: exact claim cache, evidence cache, then semantic cache. Add and benchmark one change at a time.

In [27]:
statement = "Paris is the capital of France. Germany is the capital of Berlin"
result = check_claim(statement)
print(json.dumps(result, indent=2))

{
  "claim": "Paris is the capital of France. Germany is the capital of Berlin",
  "verdict": "contradicted",
  "evidence": [
    {
      "text": "How to Differentiate Between Paris and German Cities When traveling or studying European geography, understanding the distinction between Paris and German cities is crucial: Major Cities in Germany: Berlin (capital), Munich, Hamburg, Frankfurt, Cologne, and Stuttgart. Major Cities in France: Paris, Marseille, Lyon, Toulouse, Nice, and Strasbourg.",
      "url": "https://zephyrnotes.com/blogs/france/is-paris-in-germany",
      "source_type": "search_snippet",
      "overlap": 5,
      "relevance": 1.0,
      "scores": {
        "contradiction": 0.811445415019989,
        "entailment": 0.08982151746749878,
        "neutral": 0.0987330749630928
      },
      "support_strength": 0.0898,
      "contradiction_strength": 0.8114
    }
  ],
  "errors": [],
  "timings_ms": {
    "checkability": 0.02,
    "search": 1049.75,
    "retrieval": 3744.9,
  